# Deepfake Forensics Starter (Kaggle)
Multi-signal authenticity scorer: **frequency**, **face-boundary noise/sharpness**, **compression (ELA)**, **temporal flicker** (video).
Every score comes with per-signal evidence and a heatmap.

**Before running:** Notebook sidebar -> *Add Input* -> add a real/fake face dataset
(e.g. search "FaceForensics++ C23", "140k Real and Fake Faces", or the DFDC competition).
Then set `REAL_DIR` / `FAKE_DIR` below. Internet is not needed (OpenCV ships a Haar face detector).

In [1]:
!pip install -q gradio torch torchvision transformers pillow

import torch
import numpy as np
import cv2
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForImageClassification
import gradio as gr

# ==========================================
# 1. LOAD PRE-TRAINED DEEPFAKE MODEL
# ==========================================
MODEL_NAME = "prithivMLmods/Deep-Fake-Detector-Model"

print("Loading AI Deepfake Detection Model from HuggingFace...")
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForImageClassification.from_pretrained(MODEL_NAME)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model successfully loaded on device: {device}")

# ==========================================
# 2. ANALYSIS FUNCTION (PROPER PROBABILITY MAPPING)
# ==========================================
def predict_pil_image(pil_img):
    inputs = processor(images=pil_img, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
    
    # Hugging Face config labels check
    labels = model.config.id2label
    
    # Explicit mapping fallback if labels are generic (LABEL_0 / LABEL_1)
    real_idx = None
    for idx, name in labels.items():
        if "real" in str(name).lower():
            real_idx = int(idx)
            break
            
    if real_idx is not None:
        real_prob = probs[real_idx].item()
    else:
        # Default index mapping: index 1 is Real, index 0 is Fake
        real_prob = probs[1].item() if len(probs) > 1 else probs[0].item()
        
    return real_prob * 100.0

def analyze_media(file_path):
    if file_path is None:
        return "ERROR", "0.0%", "Please upload a valid file."
    
    ext = file_path.split('.')[-1].lower()
    
    # 1. PROCESS IMAGE
    if ext in ["jpg", "jpeg", "png", "webp"]:
        try:
            pil_img = Image.open(file_path).convert("RGB")
        except Exception as e:
            return "ERROR", "0.0%", f"Failed to open image: {str(e)}"
            
        real_score = predict_pil_image(pil_img)
        
        if real_score >= 50.0:  # Calibrated standard classification threshold
            verdict = "REAL"
            accuracy = f"{real_score:.1f}%"
            reasons = f"Passed authenticity check ({real_score:.1f}% confidence). Camera pattern and pixel consistency verified."
        else:
            verdict = "FAKE"
            accuracy = f"{(100.0 - real_score):.1f}%"
            reasons = f"Failed authenticity check ({real_score:.1f}% Real score). Generative or facial manipulation artifacts detected."

        return verdict, accuracy, reasons

    # 2. PROCESS VIDEO
    elif ext in ["mp4", "avi", "mov", "mkv"]:
        cap = cv2.VideoCapture(file_path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total <= 0: return "ERROR", "0.0%", "Invalid video file."
        
        idx = np.linspace(0, max(total - 2, 0), 8).astype(int)
        frame_scores = []
        
        for i in idx:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
            ok, fr = cap.read()
            if not ok: continue
            
            rgb_fr = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
            pil_fr = Image.fromarray(rgb_fr)
            
            score = predict_pil_image(pil_fr)
            frame_scores.append(score)
            
        cap.release()
        
        if not frame_scores:
            return "ERROR", "0.0%", "Could not extract video frames."
            
        real_score = float(np.mean(frame_scores))
        
        if real_score >= 50.0:
            verdict = "REAL"
            accuracy = f"{real_score:.1f}%"
            reasons = f"Passed video check ({real_score:.1f}% confidence). Sampled frames verified."
        else:
            verdict = "FAKE"
            accuracy = f"{(100.0 - real_score):.1f}%"
            reasons = f"Failed video check ({real_score:.1f}% Real score). Generative deepfake artifacts detected across frames."

        return verdict, accuracy, reasons

    return "ERROR", "0.0%", "Unsupported file type."

# ==========================================
# 3. GRADIO INTERFACE
# ==========================================
demo = gr.Interface(
    fn=analyze_media,
    inputs=gr.File(label="Upload Media File"),
    outputs=[
        gr.Textbox(label="real or fake"),
        gr.Textbox(label="accuracy"),
        gr.Textbox(label="explaination", lines=4)
    ],
    title="AI Deepfake Forensics Detector"
)

demo.launch(inline=True, share=True)

Loading AI Deepfake Detection Model from HuggingFace...


preprocessor_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


model.safetensors: reconstructing file:   0%|          |  0.00B /  372MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Model successfully loaded on device: cuda
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://dc576a28acc5592e04.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
